### Silver - Regiões

Processada antes das demais fontes por ser referenciada por clientes,
vendedores e entregas, que representam estado/região com grafias
distintas (sigla, nome completo, com ou sem acentuação, variação de
caixa). A normalização de UF é resolvida uma única vez neste notebook e
reutilizada pelos demais.

Problemas identificados no arquivo de regiões:
- `S` e `sul` representam a mesma regional, com grafias distintas (sigla
  e nome por extenso)
- `SE` está duplicado, com `state=SP` em um registro e `state=sao paulo`
  no outro
- a linha `XX` é um registro inativo sem nome e sem estado
  (`active_flag=0`), removido da dimensão final

A granularidade adotada é de região geográfica (N/NE/CO/SE/S), nível
referenciado nas perguntas de negócio do case.

In [ ]:
%run "../00_setup/00_setup_ambiente"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.window import Window

Tabela de apoio para mapear as variações de grafia de UF identificadas em
todas as fontes para a sigla oficial e a região geográfica correspondente.

In [ ]:
map_uf_data = [
    ("AM", "AM", "Amazonas", "Norte"),
    ("BA", "BA", "Bahia", "Nordeste"),
    ("GO", "GO", "Goiás", "Centro-Oeste"),
    ("MG", "MG", "Minas Gerais", "Sudeste"),
    ("MINAS GERAIS", "MG", "Minas Gerais", "Sudeste"),
    ("PR", "PR", "Paraná", "Sul"),
    ("PARANA", "PR", "Paraná", "Sul"),
    ("PARANÁ", "PR", "Paraná", "Sul"),
    ("RJ", "RJ", "Rio de Janeiro", "Sudeste"),
    ("RIO DE JANEIRO", "RJ", "Rio de Janeiro", "Sudeste"),
    ("SC", "SC", "Santa Catarina", "Sul"),
    ("SANTA CATARINA", "SC", "Santa Catarina", "Sul"),
    ("STA CATARINA", "SC", "Santa Catarina", "Sul"),
    ("S. CATARINA", "SC", "Santa Catarina", "Sul"),
    ("SP", "SP", "São Paulo", "Sudeste"),
    ("SAO PAULO", "SP", "São Paulo", "Sudeste"),
    ("SÃO PAULO", "SP", "São Paulo", "Sudeste"),
]

df_map_uf = spark.createDataFrame(
    map_uf_data,
    StructType([
        StructField("variacao_norm", StringType(), False),
        StructField("uf_sigla", StringType(), False),
        StructField("uf_nome", StringType(), False),
        StructField("regiao_geografica", StringType(), False),
    ])
)

df_map_uf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{schema_name}.apoio_map_uf")
display(df_map_uf)

In [ ]:
def normalizar_texto(col):
    c = F.upper(F.trim(col))
    return F.translate(c, "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ", "AAAAAEEEEIIIIOOOOOUUUUC")

In [ ]:
df_regioes_bronze = spark.table(f"{schema_name}.bronze_regioes")

df_regioes_norm = (
    df_regioes_bronze
    .withColumn("state_norm", normalizar_texto(F.col("state")))
    .join(df_map_uf, F.col("state_norm") == F.col("variacao_norm"), "left")
    .filter(F.col("active_flag") == "1")  # tira a linha XX
)

Deduplicação por região geográfica: S/sul e os dois registros de SE
resultam no mesmo valor de `regiao_geografica` após o join. A sigla mais
curta é adotada como código canônico ("S", e não "sul"), preservando o
nome do gestor regional.

In [ ]:
w = Window.partitionBy("regiao_geografica").orderBy(F.length("regional_code").asc())

df_regioes_silver = (
    df_regioes_norm
    .withColumn("rk", F.row_number().over(w))
    .filter(F.col("rk") == 1)
    .select(
        F.col("regiao_geografica").alias("regiao_id"),
        F.col("regiao_geografica").alias("regiao_nome"),
        F.col("uf_sigla").alias("uf_principal"),
        F.col("manager_name").alias("gestor_regional"),
    )
)

(
    df_regioes_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.silver_regioes")
)

display(df_regioes_silver)

In [ ]:
print("bronze:", df_regioes_bronze.count(), "-> silver:", df_regioes_silver.count())
print("descartadas (active_flag=0):", df_regioes_bronze.filter(F.col("active_flag") == "0").count())